In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default='notebook'
df = pd.read_csv("WeatherEvents_Jan2016-Dec2022.csv")

In [ ]:
#Selezionare eventi atmosferici storici che sono avvenuti nell'arco del tempo del nostro dataset e mostrarli con dei grafici (magari lo spostamento di un huragano (grafico animato))

#Avendo data di inizio e data di fine posso cercare gli eventi piu lunghi e vedere cosa sono

#Grafico HeatMap che mostra l'andamento degli eventi 'SEVERE' nei vari mesi per ogni anno (sulle X i mesi, sulle Y gli anni) il colore indica se ce ne sono stati tanti o pochi

#Trasformo la colonna StartTime in una data con pandas
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])

#Prendo solo quelli severi
df_severe = df[df['Severity'] == 'Severe'].copy()
#il copy va usato perche pandas ritorna una vista del dataseto originale se dopo modifichiamo df_severe potrebbe dare errore

df_severe['Year'] = df_severe['StartTime(UTC)'].dt.year
df_severe['Month'] = df_severe['StartTime(UTC)'].dt.month
#trasformandolo in datetime di pandas ci possiamo accedere con dt e abbiamo le sue funzionalità

#Creiamo la heatmap

#Contiamo per ongi combinazione di mese-anno il numero di eventi severe (collassa tutte le righe Gennaio 2016 in una sola contando il numero di occorrenze)
heatmap = df_severe.groupby(['Year','Month']).size().reset_index(name='Count')
#Quindi dopo qui abbiamo le righe singole che sono Gennaio 2016 / febbraio 2016 ... agosto 2016 e per ognuna di queste righe uniche abbiamo il numero di count 

#Creiamo una tabella pivot da passare alla heatmap
pivot_table = heatmap.pivot(index='Year', columns='Month', values='Count').fillna(0)

fig = px.imshow(pivot_table, 
                title="Stagionalità e Frequenza degli Eventi Estremi (2016-2022)",
                labels=dict(x="Mese", y="Anno", color="N eventi Estremi"),
                x=['Gen', 'Feb', 'Mar', 'Apr', 'Mag', 'Giu', 'Lug', 'Ago', 'Set', 'Ott', 'Nov', 'Dic']
                );

fig.show()


In [ ]:
#Analisi Uragano Harvey
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])

df_texas = df[df['State'] == 'TX'].copy()
#Facciamo la copy per lo stesso motivo di prima

df_texas['Year'] = df_texas['StartTime(UTC)'].dt.year
df_texas['Month'] = df_texas['StartTime(UTC)'].dt.month
df_texas['Day'] = df_texas['StartTime(UTC)'].dt.day

df_texas = df_texas[(df_texas['Year']==2017) & (df_texas['Month'] == 8)]

df_prec = df_texas[df_texas['Precipitation(in)'] > 0]

df_prec['Precipitation(mm)'] = df_prec['Precipitation(in)'] * 25.4;

fig = px.box(df_prec,
            x='Day',
            y='Precipitation(mm)',
            title="Distribuzione delle Precipitazioni Giornaliere in Texas - Uragano Harvey (Agosto 2017)",
            labels={'Day': 'Giorno', 'Precipitation(in)': 'Precipitazioni (pollici)'},
            )
fig.update_layout(xaxis=dict(tickmode='linear', dtick=1))

fig.show()



In [ ]:
# 1. Filtra per Agosto 2017
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'])
df_agosto_2017 = df[(df['StartTime(UTC)'].dt.year == 2017) & 
                    (df['StartTime(UTC)'].dt.month == 8) & 
                    (df['Precipitation(in)'] > 0)].copy()

# 2. Converti in millimetri
df_agosto_2017['Precipitation(mm)'] = df_agosto_2017['Precipitation(in)'] * 25.4

#Anomalia trovata nello stato di NY 1500mm (Impossibile) e 520mm (Impossibile), ho controllato nessun evento rilevante quel giorno

# 3. Abbassa la soglia di pulizia per escludere l'ulteriore anomalia di 520 mm
df_agosto_2017 = df_agosto_2017[df_agosto_2017['Precipitation(mm)'] <= 450]

# 4. Trova il PICCO MASSIMO per ogni Stato sui dati puliti
precipitazioni_stato = df_agosto_2017.groupby('State')['Precipitation(mm)'].max().reset_index()

# 5. Genera la Mappa Coropletica
fig = px.choropleth(
    precipitazioni_stato,
    locations='State',                 
    locationmode='USA-states',         
    color='Precipitation(mm)',         
    color_continuous_scale='Reds',
    range_color=[0, 430],              # Forza la scala dei colori per non farsi sfasare da eventuali altri piccoli outlier
    scope='usa',                       
    title="Picco Massimo di Precipitazioni per Stato negli USA (Agosto 2017)",
    labels={'Precipitation(mm)': 'Precipitazione Massima (mm)'}
)

fig.show()